In [ ]:
from ppp_paths import dataint


In [ ]:
import pandas as pd
from metaphone import doublemetaphone
from fuzzywuzzy import fuzz
import re
import numpy as np
import unicodedata

In [ ]:
possiblepairsauthorsinventors = pd.read_parquet(dataint + 'possiblepairsauthorsinventors.parquet')

In [ ]:
#return a ratio of similarity of letters between two strings (to handle in the first names errors)
def match_ratio(string1,string2):
    return fuzz.ratio(string1, string2)

In [ ]:
#return 4 if string1 and string2 are the same
#return 3 if string1 and string2 sound the same
#otherwise, return less

def metaphone(string1,string2):
    if string1==string2:
        return 4
    tuple1 = doublemetaphone(string1)
    tuple2 = doublemetaphone(string2)
    if tuple1[0] == tuple2[0]:
        return 3
    elif tuple1[0] == tuple2[1] or tuple1[1] == tuple2[0]:
        return 2
    elif tuple1[1] == tuple2[1]:
        return 1
    else:
        return 0

In [ ]:
## return 1 if name1 and name2 potentially represent the same individual
## else return 0

def comparison(name1 , name2):
    
    #if there is no first name, retrun it's a match
    if name1 == "" or name2 == "":
        return 1
    
    #if some first names exist:
    list_name1 = name1.split()
    list_name2 = name2.split()
    
    #minimum number of first names to match
    number_match = min( len(list_name1) , len(list_name2))
    
    #for each name, check if there is a match
    count_match = 0
    for elem1 in list_name1:
        match = 0
        
        #if we just have the initial:
        if len(elem1) == 1:
            for elem2 in list_name2:
                if elem1[0] == elem2[0]:
                    match = 1
        else:
            for elem2 in list_name2:
                #if we just have the initial:
                if len(elem2) == 1 and elem1[0] == elem2[0]:
                    match = 1
                    
                #if elem1 and elem2 are entire first names that sound the same and have a ratio of common letters higher thsan 85%, it's a match
                elif len(elem2) > 1 and (metaphone(elem1,elem2) > 2 or match_ratio(elem1 , elem2) > 85 ) :
                    match = 1
                    
        #count the number of first names that match    
        count_match += match
        
    #check if we have enough first names that match 
    if count_match < number_match:
        return 0
    else:
        return 1



In [ ]:
## get number of in common authors

def number_of_in_common_authors(inventors , authors ):
    

    
    #count the number of names in common, and store the names in common
    count = 0
    list_in_common_authors = []

    
    for name_inventor in inventors:
        
        for name_author in authors:
                        
            
            if len(set(name_inventor.split()) & set( name_author.split())) > 0:
            
                match = comparison(name_author , name_inventor)

                #if the first names match, we store the first names that are matching and their index 
                if match == 1:
                    count += 1 
                    list_in_common_authors.append(name_author + "-" + name_inventor)
                    


    #return 1) the number of names in common, 2) the list of names in common, 3) their index 
    return  count ,  list_in_common_authors 


In [ ]:
def notnormalize(data):
    return data 

In [ ]:
import pandas as pd

# # Example DataFrame
# smallhope = pd.DataFrame({
#     'patent_id': ['P12345', 'P12346'],
#     'inventors': ['John Smith#Jane Doe', 'Bob Jones#Alice Brown'],
#     'authors': ['Jane Doe#Mike Lee', 'Bob Jones#Eve White']
# })

# # Example functions
# def normalize(name):
#     return name.strip().lower()

# def number_of_in_common_authors(inventors, authors):
#     common = len(set(inventors) & set(authors))
#     return (common, [])

# Vectorized processing

possiblepairsauthorsinventors['inventors_list'] = possiblepairsauthorsinventors['inventors'].apply(
    lambda x: [notnormalize(elem) for elem in set(x.split("#"))]
)
possiblepairsauthorsinventors['authors_list'] = possiblepairsauthorsinventors['authors'].apply(
    lambda x: [notnormalize(elem) for elem in set(x.split("#"))]
)

# possiblepairsauthorsinventors.rename(columns={'inventors' : 'inventors_list', 'authors' : 'authors_list'}, inplace=True)

In [ ]:

# Compute common authors
possiblepairsauthorsinventors['authors in common'] = possiblepairsauthorsinventors.apply(
    lambda row: number_of_in_common_authors(row['inventors_list'], row['authors_list'])[0], 
    axis=1
)

# Compute proportion
possiblepairsauthorsinventors['proportion inventors'] = possiblepairsauthorsinventors.apply(
    lambda row: 100 * (row['authors in common'] / len(row['inventors_list'])) 
    if row['inventors_list'] else 0, 
    axis=1
)

# Drop temporary columns
possiblepairsauthorsinventors = possiblepairsauthorsinventors.drop(['inventors_list', 'authors_list'], axis=1)

# Print results
print("\nFinal DataFrame:")
print(possiblepairsauthorsinventors)

In [ ]:
possiblepairsauthorsinventors.columns

In [ ]:
subset = possiblepairsauthorsinventors[0:1000]
subset.to_csv(dataint + 'SUBSETpossiblepairscontributoroverlap.csv',  index=False)

In [ ]:
possiblepairsauthorsinventors = possiblepairsauthorsinventors[['magid', 'patent_id', 'authors in common', 'proportion inventors']]

In [ ]:
possiblepairsauthorsinventors.to_parquet(dataint + 'possiblepairs_contributoroverlap.parquet',  index=False, compression='snappy')